# Лабораторна робота 5 — Логістична регресія для аналізу тональності

**Набори даних:** `amazon_baby_subset.csv`, `important_words.json`  
**Обмеження:** scikit-learn-класифікатори **не дозволені** для базових завдань.

## Налаштування

In [1]:
import sys
#!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [2]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline


## Теоретичне підґрунтя

Сигмоїдна функція:
```
P(y = +1 | x, w) = 1 / (1 + exp(−wᵀ h(x)))
```
Похідна логарифму правдоподібності відносно wⱼ:
```
d(ll)/dwⱼ = dot( hⱼ,  1[y=+1] − P(y=+1|x,w) )
```
де 1[y=+1] = 1, якщо мітка +1, інакше 0.

---
## Завдання 1 — Підготовка ознак

1. Завантажте `amazon_baby_subset.csv`. Видаліть рядки з відсутніми відгуками. Вилучіть відгуки з `rating == 3`. Створіть стовпець `sentiment`: **+1** якщо `rating >= 4`, інакше **−1**.
2. Завантажте `important_words.json` (193 слова). Для кожного слова додайте стовпець до DataFrame із підрахунком його входжень у очищений текст відгуку.
3. Повідомте, скільки відгуків залишилось та який баланс класів.

In [3]:
# Завантаження набору даних
products = pd.read_csv('amazon_baby_subset.csv')

# Видаліть рядки з відсутніми відгуками та нейтральними рейтингами
products = products[(products['review'] != '') & (products['rating'] != 3)]

# Створіть стовпець sentiment: +1 якщо рейтинг >= 4, інакше -1
products.loc[products['rating'] >= 4, 'sentiment'] = 1
products.loc[products['rating'] < 4, 'sentiment'] = -1

print(f'Усього відгуків : {len(products)}')
print(f'Позитивні (+1)  : {(products["sentiment"] == 1).sum()}')
print(f'Негативні (-1)  : {(products["sentiment"] == -1).sum()}')


Усього відгуків : 17386
Позитивні (+1)  : 17385
Негативні (-1)  : 0


In [4]:
# Завантаження важливих слів
with open('important_words.json') as f:
    important_words = json.load(f)
print(f'Розмір словника: {len(important_words)} слів')


Розмір словника: 193 слів


In [5]:
# Очищення тексту відгуків (видалення пунктуації, приведення до нижнього регістру)
import string
products['review_clean'] = (
    products['review']
    .fillna('')
    .str.replace(f'[{string.punctuation}]', '', regex=True)
    .str.lower()
)

# Додайте стовпець підрахунку слів для кожного важливого слова
for word in important_words:
    products[word] = products['review_clean'].apply(
        lambda text: text.split().count(word)
    )

print(products)

#print('Приклад підрахунку слів:')
#products[important_words[:5]].head(3)


<ipython-input-5-65d0132eeb13>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  products[word] = products['review_clean'].apply(
<ipython-input-5-65d0132eeb13>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  products[word] = products['review_clean'].apply(
<ipython-input-5-65d0132eeb13>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented

                                                    name  \
0      Stop Pacifier Sucking without tears with Thumb...   
1        Nature's Lullabies Second Year Sticker Calendar   
2        Nature's Lullabies Second Year Sticker Calendar   
3                            Lamaze Peekaboo, I Love You   
4      SoftPlay Peek-A-Boo Where's Elmo A Children's ...   
...                                                  ...   
17381  Fisher-Price Healthy Care Booster Seat, Green/...   
17382  Fisher-Price Healthy Care Booster Seat, Green/...   
17383  Luvable Friends Flannel Fitted Crib Sheet, Yel...   
17384  Luvable Friends Flannel Fitted Crib Sheet, Yel...   
17385  Luvable Friends Flannel Fitted Crib Sheet, Yel...   

                                                  review  rating  sentiment  \
0      All of my kids have cried non-stop when I trie...     5.0        1.0   
1      We wanted to get something to keep track of ou...     5.0        1.0   
2      My daughter had her 1st baby over a

---
## Завдання 2 — Побудова матриці ознак

Реалізуйте `get_feature_matrix(df, word_list)`, яка:
1. Створює масив NumPy зі стовпцем одиниць (вільний член), за яким іде по одному стовпцю для кожного слова зі списку.
2. Повертає `(feature_matrix, sentiment_array)`, де `sentiment_array` містить +1 або −1.

Перевірка: `feature_matrix` має форму `(N, 194)`.

In [6]:
def get_feature_matrix(df, word_list):
    """
    Будує матрицю ознак та вектор міток тональності.

    Повертає
    -------
    feature_matrix  : np.ndarray, shape (n, len(word_list)+1)
    sentiment_array : np.ndarray, shape (n,)  значення {+1, -1}
    """
    # Intercept column of 1s
    intercept = np.ones((len(df), 1))
    # Word feature columns
    words = df[word_list].to_numpy()
    feature_matrix = np.append(intercept, words, axis = 1)
    
    sentiment_array = df['sentiment'].to_numpy()
    return feature_matrix, sentiment_array


In [7]:
feature_matrix, sentiment = get_feature_matrix(products, important_words)
print(f'Розмір feature_matrix: {feature_matrix.shape}')  # очікується (N, 194)


Розмір feature_matrix: (17386, 194)


---
## Завдання 3 — Сигмоїдна функція та передбачення

Реалізуйте `predict_probability(feature_matrix, coefficients)`, що обчислює сигмоїдну функцію для кожного рядка. Результат — масив NumPy зі значеннями у (0, 1).

In [8]:
def predict_probability(feature_matrix, coefficients):
    """
    Обчислює P(y = +1 | x, w) для кожного рядка.

    Повертає
    -------
    probabilities : np.ndarray, shape (n,), значення у (0, 1)
    """
    # Compute the linear score for each example
    score = np.dot(feature_matrix, coefficients)

    # Apply sigmoid
    probabilities = 1 / (1 + np.exp(-score))

    return probabilities


### Перевірка — при нульових вхідних даних кожне передбачення має дорівнювати 0.5

In [9]:
zero_coeffs = np.zeros(feature_matrix.shape[1])
test_probs  = predict_probability(feature_matrix, zero_coeffs)
print(f'Усі передбачення 0.5: {np.allclose(test_probs, 0.5)}')


Усі передбачення 0.5: True


---
## Завдання 4 — Градієнтний підйом

Реалізуйте `logistic_regression(feature_matrix, sentiment, initial_coefficients, step_size, max_iter)`. На кожній ітерації:
1. Обчислюйте передбачення за допомогою `predict_probability()`.
2. Обчислюйте `errors = 1[y=+1] − predictions`.
3. Для кожного коефіцієнта j: `derivative = dot(feature_j, errors)`, потім `coeff[j] += step_size · derivative`.

Запустіть з: `initial_coefficients = np.zeros(194)`, `step_size = 1e-7`, `max_iter = 301`. Виводьте логарифм правдоподібності кожні 50 ітерацій — він має зростати монотонно.

In [10]:
def compute_log_likelihood(feature_matrix, sentiment, coefficients):
    """Допоміжна функція (надана) — обчислює логарифм правдоподібності для моніторингу."""
    indicator = (sentiment == +1).astype(float)
    scores    = np.dot(feature_matrix, coefficients)
    ll        = np.sum(
        indicator * scores - np.log(1.0 + np.exp(scores))
    )
    return ll


In [11]:
def logistic_regression(feature_matrix, sentiment,
                        initial_coefficients, step_size, max_iter):
    """
    Навчає ваги логістичної регресії методом градієнтного підйому.

    Повертає
    -------
    coefficients : np.ndarray, shape (n_features,)
    """
    coefficients = np.array(initial_coefficients, dtype=float)
    indicator    = (sentiment == +1).astype(float)   # 1 if positive, 0 otherwise

    for itr in range(max_iter):
        # 1. Compute predictions P(y=+1 | x, w)
        predictions = predict_probability(feature_matrix, coefficients)

        # 2. Compute errors = indicator(y=+1) - predictions
        errors = indicator - predictions

        # 3. Update every coefficient
        for j in range(len(coefficients)):
            derivative = np.dot(feature_matrix[:, j], errors)
            coefficients[j] += step_size * derivative

        # Print log-likelihood every 50 iterations
        if itr % 50 == 0:
            ll = compute_log_likelihood(feature_matrix, sentiment, coefficients)
            print(f'Ітерація {itr:4d}  |  логарифм правдоподібності: {ll:.4f}')

    return coefficients


### Запуск моделі

In [12]:
coefficients = logistic_regression(
    feature_matrix, sentiment,
    initial_coefficients=np.zeros(194),
    step_size=1e-7,
    max_iter=301
)


Ітерація    0  |  логарифм правдоподібності: -12026.9567
Ітерація   50  |  логарифм правдоподібності: -10933.3758
Ітерація  100  |  логарифм правдоподібності: -10025.0205
Ітерація  150  |  логарифм правдоподібності: -9260.3689
Ітерація  200  |  логарифм правдоподібності: -8608.1748
Ітерація  250  |  логарифм правдоподібності: -8045.2484
Ітерація  300  |  логарифм правдоподібності: -7554.2621


### Точність класифікації та базовий рівень

In [13]:
# Передбачте мітки класів (+1 якщо score > 0, інакше -1)
scores          = np.dot(feature_matrix, coefficients)
predictions     = np.where(scores > 0, +1, -1)

# Точність моделі
model_accuracy  = np.mean(predictions == sentiment)
print(f'Точність моделі   : {model_accuracy:.4f}')

# Базовий рівень мажоритарного класу
majority_class  = +1 if (sentiment == +1).sum() >= (sentiment == -1).sum() else -1
baseline_acc    = np.mean(majority_class == sentiment)
print(f'Базовий рівень    : {baseline_acc:.4f}  (завжди передбачає {majority_class})')


Точність моделі   : 0.9999
Базовий рівень    : 0.9999  (завжди передбачає 1)


---
## ✨ Бонус — Інтерпретація моделі

Зіставте кожне слово з його навченим коефіцієнтом. Виведіть 10 слів з найбільшими коефіцієнтами та 10 слів з найменшими.

Для одного слова з кожного списку знайдіть відгук у наборі даних, що його містить, і вкажіть передбачувану ймовірність моделі.

In [ ]:
# Бонус — аналіз коефіцієнтів
# Підказка: створіть DataFrame зі стовпцями ['word', 'coefficient']
# ВАШ КОД ТУТ
raise NotImplementedError

In [ ]:
# Бонус — приклад відгуку з передбачуваною ймовірністю
# ВАШ КОД ТУТ
raise NotImplementedError

**Найбільш позитивні слова:** *перелічіть їх тут*  
**Найбільш негативні слова:** *перелічіть їх тут*  
**Спостереження:** *прокоментуйте по одному слову з кожного списку*